# RecSys 2026 — Music-CRS: Exploratory Data Analysis

**Challenge:** Conversational Music Recommendation System  
**Dataset:** TalkPlayData-Challenge (`talkpl-ai/talkplay-data-challenge`)  

This notebook explores the dataset structure:
- Conversation statistics (turns, session lengths)
- Track metadata distribution (genres, artists, release years)
- User profile demographics
- Ground truth relevance analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)

from dotenv import load_dotenv
load_dotenv('../.env')
print('Setup complete ✅')

## 1. Load Dataset

In [ ]:
from src.utils.data_loader import load_split, load_track_metadata, load_user_profiles

# Load train split
train_ds = load_split('train', data_dir=Path('../data/raw'))
print(f'Train dataset: {train_ds}')
print(f'Features: {train_ds.features}')

In [ ]:
train_df = train_ds.to_pandas()
print(f'Shape: {train_df.shape}')
train_df.head(3)

## 2. Conversation Statistics

In [ ]:
# Sessions and turns
session_turn_counts = train_df.groupby('session_id')['turn_number'].max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(session_turn_counts, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Turns per Session')
axes[0].set_xlabel('Number of Turns')
axes[0].set_ylabel('Count')

utterance_lengths = train_df['utterance'].str.split().str.len()
axes[1].hist(utterance_lengths, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Utterance Lengths (tokens)')
axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'Total sessions: {train_df["session_id"].nunique():,}')
print(f'Total turns: {len(train_df):,}')
print(f'Avg turns/session: {session_turn_counts.mean():.1f}')

## 3. Track Metadata Analysis

In [ ]:
tracks_df = load_track_metadata(data_dir=Path('../data/raw'))
print(f'Track catalog size: {len(tracks_df):,}')
print(f'Columns: {list(tracks_df.columns)}')
tracks_df.head()

In [ ]:
# Top artists
top_artists = tracks_df['artist_name'].value_counts().head(20)

plt.figure(figsize=(12, 5))
top_artists.plot(kind='barh', color='mediumseagreen')
plt.title('Top 20 Artists by Track Count')
plt.xlabel('Number of Tracks')
plt.tight_layout()
plt.show()

In [ ]:
# Tag / Genre distribution
if 'tags' in tracks_df.columns:
    all_tags = tracks_df['tags'].dropna().str.split(',').explode().str.strip()
    top_tags = all_tags.value_counts().head(25)

    plt.figure(figsize=(12, 6))
    top_tags.plot(kind='bar', color='mediumpurple')
    plt.title('Top 25 Genre/Mood Tags')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 4. User Profile Analysis

In [ ]:
users_df = load_user_profiles(data_dir=Path('../data/raw'))
print(f'Total users: {len(users_df):,}')
users_df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gender
users_df['gender'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%')
axes[0].set_title('Gender Distribution')

# Age
axes[1].hist(users_df['age'].dropna(), bins=20, color='dodgerblue', edgecolor='white')
axes[1].set_title('Age Distribution')
axes[1].set_xlabel('Age')

# Top countries
users_df['country'].value_counts().head(10).plot(kind='barh', ax=axes[2], color='tomato')
axes[2].set_title('Top 10 Countries')

plt.tight_layout()
plt.show()

## 5. Sample Conversations

In [ ]:
# Show a full sample conversation
sample_session = train_df['session_id'].iloc[0]
session_turns = train_df[train_df['session_id'] == sample_session].sort_values('turn_number')

print(f'Session: {sample_session}')
print('='*60)
for _, row in session_turns.iterrows():
    print(f'Turn {row["turn_number"]}: {row["utterance"]}')
    print()